# Codebook vs Uniform 4/3/2-bit — SOTA-aligned accuracy sweep

Produces the accuracy table for the **codebook-gather** paper on the protocol the
quantization field uses (GPTQ / AWQ / SqueezeLLM / QuIP#). Because the Apple-AMX kernel
is **bit-exact** vs `A·dequant(W)`, perplexity computed here in PyTorch equals what the
kernel produces — so this notebook *is* the accuracy table; the kernel supplies speed/energy.

**What it does**
- Models: any HF causal LM (Llama-2-7B / Mistral-7B on a GPU; TinyLlama to smoke-test).
- Calibration: **C4**, 128×2048 = 262k tokens (full-rank Hessian; no WikiText leakage).
- Eval: **WikiText-2** (and optionally C4) perplexity at seq-len 2048.
- Methods: `uniform` / `codebook (NF)` / `NF+GPTQ`, at bit-widths **4, 3, 2**.
- Grouping: per-group 128 (and 64), the standard knob.
- GPTQ: blocked + activation-order + Cholesky-inverse error feedback (the real algorithm).

**Cost notes** (see the paper's plan)
- 7B fp16 ≈ 14 GB → fits a **24 GB** card (RTX 4090 / A10 / L4, ~\$0.30–0.50/hr) or Kaggle's
  free **2×T4 (30 GB)**. You do **not** need an A100 for 7B.
- Only `NF+GPTQ` needs calibration; `uniform`/`codebook` are instant.
- Results checkpoint to CSV after every cell, so a spot-instance kill never loses work.

**Before running:** set a GPU runtime. Llama-2 is gated — accept the license on HuggingFace
and paste a token in the login cell.

In [ ]:
# transformers/accelerate pinned to versions compatible with torch 2.3.1 (the Pascal/P100
# build installed below). Newer transformers require torch >= 2.6 and break on P100.
!pip -q install "transformers==4.44.2" "accelerate==0.34.2" "datasets>=2.20,<3" sentencepiece protobuf huggingface_hub "lm-eval==0.4.3" 

### GPU compatibility fix (Kaggle free GPU)
Kaggle's API grants a **P100** (compute capability sm_60), but its stock PyTorch dropped
Pascal support (`no kernel image` error). On P100 only, install `torch==2.3.1` (cu121, the
last build with sm_60) **before** importing torch. No-op on T4/newer GPUs.

In [ ]:
import subprocess, sys
smi = subprocess.run(["nvidia-smi","-L"], capture_output=True, text=True).stdout
print("GPU:", smi.strip())
if "P100" in smi:
    print("P100 (sm_60): installing torch 2.3.1 cu121 (with deps -> cuDNN8) before importing torch...")
    r = subprocess.run([sys.executable,"-m","pip","install","-q","torch==2.3.1",
                        "--index-url","https://download.pytorch.org/whl/cu121"], capture_output=True, text=True)
    print("torch install rc", r.returncode, (r.stderr[-300:] if r.returncode else ""))

In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # reduce fragmentation across model reloads
import torch, numpy as np, math, csv, time
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "(CPU)")
DEV = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEV == "cuda" else torch.float32
_T0 = time.time()
def EL(): s=int(time.time()-_T0); return f"[{s//60}m{s%60:02d}s]"   # elapsed stamp for progress logs

### Hugging Face login (needed for gated models like Llama-2). Skip for TinyLlama/GPT-2.

In [ ]:
from huggingface_hub import login
# login()   # <-- uncomment and run; paste your token when prompted

### Configuration — edit these knobs

In [ ]:
CONFIG = dict(
    models      = [                             # runs ALL of these in one session (one CSV)
        "TinyLlama/TinyLlama-1.1B-Chat-v1.0",   # small sanity model (minutes)
        "meta-llama/Llama-2-7b-hf",             # headline (needs the license accepted)
        # "mistralai/Mistral-7B-v0.1",
    ],
    bits        = [4, 3, 2],                     # bit-widths to sweep
    groups      = [128, 64],                     # per-group sizes
    ctx         = 2048,                          # eval/calib window (auto-clamped to model max)
    calib_source= "c4",                          # "c4" (SOTA) or "wikitext" (quick, leaky)
    calib_tokens= 262144,                        # 128 x 2048 (SOTA). Lower to go faster.
    eval_tokens = 0,                             # 0 = full WikiText-2 test
    eval_c4     = True,                          # also report C4 perplexity
    # downstream zero-shot (lm-eval-harness) for HEADLINE configs only (cost control):
    ds_tasks    = ["piqa","arc_easy","arc_challenge","hellaswag","winogrande","boolq"],
    ds_limit    = 0,                             # 0 = full test set; set e.g. 500 to sample
    ds_methods  = ["uniform","nf+gptq","km+gptq"],  # which methods get downstream
    ds_groups   = [128],                         # only g128 gets downstream
    out_csv     = "results.csv", ds_csv = "downstream.csv",
)
CONFIG

### Codebooks for any bit-width
Fixed **NormalFloat** levels (great at 4-bit) and per-group **k-means** (needed at 2-bit,
where a fixed grid can't adapt — profiled: fixed NF *loses* to uniform at 2-bit, k-means
beats it ~40%).

In [ ]:
NF4 = [-1.0,-0.6961928,-0.5250731,-0.3949175,-0.2844414,-0.1847734,-0.0910500,0.0,
       0.0795803,0.1609302,0.2461123,0.3379152,0.4407098,0.5626170,0.7229568,1.0]

def nf_levels(bits, device, dtype=torch.float32):
    """NormalFloat: 2^bits normal-quantile levels, [-1,1]. Exactly QLoRA NF4 at bits==4;
    otherwise QLoRA's construction generalized (asymmetric halves sharing an exact zero)."""
    n = 2 ** bits
    if bits == 4:
        lv = torch.tensor(NF4, dtype=torch.float64)
    else:
        offset = 1 - 0.5 * (1/(2*n) + 1/(2*(n-1))); half = n // 2
        pos = torch.special.ndtri(torch.linspace(offset, 0.5, half+1, dtype=torch.float64)[:-1])
        neg = -torch.special.ndtri(torch.linspace(offset, 0.5, half, dtype=torch.float64)[:-1])
        lv = torch.cat([neg, torch.zeros(1, dtype=torch.float64), pos]).sort().values
    return (lv / lv.abs().max()).to(dtype).to(device)

### Quantizers: uniform, fixed-NF codebook, k-means codebook, and blocked GPTQ

In [ ]:
@torch.no_grad()
def quant_uniform(W, bits, group):
    n = 2 ** bits; out, inn = W.shape; Q = W.clone()
    for c0 in range(0, inn, group):
        c1 = min(c0 + group, inn); g = W[:, c0:c1]
        lo = g.amin(1, keepdim=True); hi = g.amax(1, keepdim=True)
        st = (hi - lo) / (n - 1) + 1e-12
        Q[:, c0:c1] = lo + torch.clamp(torch.round((g - lo) / st), 0, n - 1) * st
    return Q

@torch.no_grad()
def quant_codebook(W, bits, group, levels):
    out, inn = W.shape; Q = W.clone()
    for c0 in range(0, inn, group):
        c1 = min(c0 + group, inn); g = W[:, c0:c1]
        s = g.abs().amax(1, keepdim=True).clamp_min(1e-8)
        idx = (g.unsqueeze(-1) / s.unsqueeze(-1) - levels).abs().argmin(-1)
        Q[:, c0:c1] = levels[idx] * s
    return Q

@torch.no_grad()
def quant_kmeans(W, bits, group, iters=12):
    n = 2 ** bits; out, inn = W.shape; Q = W.clone()
    for c0 in range(0, inn, group):
        c1 = min(c0 + group, inn); g = W[:, c0:c1]
        s = g.abs().amax(1, keepdim=True).clamp_min(1e-8); x = g / s; gw = x.shape[1]
        qi = ((torch.arange(n, device=W.device)+0.5)*gw/n).long().clamp(max=gw-1)
        c = torch.sort(x, 1).values[:, qi]
        for _ in range(iters):
            a = (x.unsqueeze(-1) - c.unsqueeze(1)).abs().argmin(-1)
            for e in range(n):
                m = (a == e); cnt = m.sum(1); nz = cnt > 0
                c[nz, e] = (x*m).sum(1)[nz]/cnt[nz]
        a = (x.unsqueeze(-1) - c.unsqueeze(1)).abs().argmin(-1)
        Q[:, c0:c1] = torch.gather(c, 1, a) * s
    return Q

def _fit_grid(cols, bits, levels, codebook, dev, iters=8):
    n = 2 ** bits; s = cols.abs().amax(1, keepdim=True).clamp_min(1e-8)
    if codebook == "nf": return s * levels[None, :]
    x = cols / s; gw = x.shape[1]
    qi = ((torch.arange(n, device=dev)+0.5)*gw/n).long().clamp(max=gw-1)
    c = torch.sort(x, 1).values[:, qi]
    for _ in range(iters):
        a = (x.unsqueeze(-1) - c.unsqueeze(1)).abs().argmin(-1)
        for e in range(n):
            m = (a == e); cnt = m.sum(1); nz = cnt > 0
            c[nz, e] = (x*m).sum(1)[nz]/cnt[nz]
    return c * s

@torch.no_grad()
def gptq(W, H, bits, group, levels, codebook="nf", actorder=True, blocksize=128, percdamp=0.01):
    W = W.clone().float(); H = H.clone().float(); out, inn = W.shape
    dev = W.device; levels = levels.to(dev).float()
    dead = torch.diag(H) == 0; H[dead, dead] = 1; W[:, dead] = 0
    if actorder:
        perm = torch.argsort(torch.diag(H), descending=True)
        W = W[:, perm]; H = H[perm][:, perm]; inv = torch.argsort(perm)
    i = torch.arange(inn, device=dev); H[i, i] += percdamp * torch.diag(H).mean()
    Hinv = torch.linalg.cholesky(torch.cholesky_inverse(torch.linalg.cholesky(H)), upper=True)
    Q = torch.zeros_like(W); grid = None
    for a in range(0, inn, blocksize):
        b = min(a + blocksize, inn)
        W1 = W[:, a:b].clone(); Q1 = torch.zeros_like(W1); E = torch.zeros_like(W1); Hi = Hinv[a:b, a:b]
        for j in range(b - a):
            col = a + j
            if (col % group) == 0:
                grid = _fit_grid(W[:, col:min(col+group, inn)], bits, levels, codebook, dev)
            w = W1[:, j]; qi = (w[:, None] - grid).abs().argmin(1)
            q = torch.gather(grid, 1, qi[:, None]).squeeze(1)
            Q1[:, j] = q; err = (w - q) / Hi[j, j]; E[:, j] = err
            if j + 1 < b - a: W1[:, j + 1:] -= err[:, None] * Hi[j, j + 1:][None, :]
        Q[:, a:b] = Q1
        if b < inn: W[:, b:] -= E @ Hinv[a:b, b:]
    return (Q[:, inv] if actorder else Q).to(W.dtype)

### Model plumbing: pick layers, collect Hessians, apply quantizers, perplexity

In [ ]:
from transformers.pytorch_utils import Conv1D
def targets(model):
    SKIP = ("lm_head","embed","wte","wpe","shared","embed_tokens")
    for n, m in model.named_modules():
        if any(s in n for s in SKIP): continue
        if isinstance(m, (torch.nn.Linear, Conv1D)) and m.weight.ndim == 2 and min(m.weight.shape) >= 64:
            yield n, m, isinstance(m, Conv1D)

@torch.no_grad()
def quantize_rtn(model, method, bits, group, dev):
    levels = nf_levels(bits, dev) if method == "codebook" else None
    seen = set()
    for _, m, conv in targets(model):
        if id(m.weight) in seen: continue
        seen.add(id(m.weight))
        W = (m.weight.data.t().contiguous() if conv else m.weight.data).to(dev).float()
        if method == "uniform":  Q = quant_uniform(W, bits, group)
        elif method == "kmeans": Q = quant_kmeans(W, bits, group)
        else:                    Q = quant_codebook(W, bits, group, levels)
        m.weight.copy_((Q.t().contiguous() if conv else Q).to(m.weight.dtype))

@torch.no_grad()
def _hess_bytes(model):   # fp32 bytes for all unique-weight Hessians (inn x inn)
    seen = set(); tot = 0
    for _, m, conv in targets(model):
        if id(m.weight) in seen: continue
        seen.add(id(m.weight))
        inn = m.weight.shape[0] if conv else m.weight.shape[1]
        tot += inn * inn * 4
    return tot

@torch.no_grad()
def collect_hessians(model, calib, ctx, dev, hess_dev=None):
    # Accumulate each layer's Hessian IN PLACE. Prefer the GPU (fast: no PCIe, in-place
    # add_) when all Hessians + model comfortably fit in VRAM; else fall back to CPU
    # accumulation (slower, but unbounded) -- e.g. 13B, whose ~56GB of Hessians won't
    # co-reside with the model on an 80GB card. The returned dict is always CPU-resident
    # so the quantize phase can stream one layer at a time (see quantize_gptq). This is
    # what keeps the GPTQ phase from the OOM/kernel-death that a whole-dict GPU move hit.
    if hess_dev is None:
        if dev == "cuda":
            free, _ = torch.cuda.mem_get_info()
            hess_dev = "cuda" if _hess_bytes(model) + 5 * 1024**3 < free else "cpu"
        else:
            hess_dev = "cpu"
    print(f"    {EL()} Hessians on {hess_dev} (~{_hess_bytes(model)/1024**3:.1f}GB fp32)", flush=True)
    H, cnt, hk = {}, {}, []          # keyed by layer NAME (survives model reloads)
    def mk(name, m):
        def h(mod, args):
            x = args[0].detach().float().reshape(-1, args[0].shape[-1])
            g = x.t() @ x
            if name not in H: H[name] = torch.zeros_like(g, device=hess_dev)
            H[name].add_(g.to(hess_dev, non_blocking=True)); cnt[name] = x.shape[0] + cnt.get(name, 0)
        return m.register_forward_pre_hook(h)
    for name, m, _ in targets(model): hk.append(mk(name, m))
    nb = calib.size(1) // ctx
    for i in range(nb):
        model(calib[:, i*ctx:(i+1)*ctx].to(dev))
        if (i+1) % 8 == 0 or i+1 == nb: print(f"    {EL()} hessian calib pass {i+1}/{nb}", flush=True)
    for h in hk: h.remove()
    out = {name: (H[name] / max(cnt[name], 1)).cpu() for name in H}   # CPU fp32 dict
    H.clear()
    if dev == "cuda": torch.cuda.empty_cache()
    return out

@torch.no_grad()
def quantize_gptq(model, hess, bits, group, dev, codebook="nf"):
    # Stream ONE Hessian at a time CPU->GPU, quantize that layer, free it. Never
    # holds more than a single layer's Hessian resident on the GPU (was: the whole
    # ~40GB dict moved to GPU at once, on top of the reloaded model -> OOM/abort).
    levels = nf_levels(bits, dev); seen = set()
    layers = [(n,m,c) for n,m,c in targets(model) if n in hess]
    tot = len(layers); done = 0
    for name, m, conv in layers:
        if id(m.weight) in seen: continue
        seen.add(id(m.weight))
        W = (m.weight.data.t().contiguous() if conv else m.weight.data).to(dev).float()
        H = hess[name].to(dev)
        Q = gptq(W, H, bits, group, levels, codebook=codebook)
        m.weight.copy_((Q.t().contiguous() if conv else Q).to(m.weight.dtype))
        del W, H, Q
        if dev == "cuda": torch.cuda.empty_cache()
        done += 1
        if done % 4 == 0 or done == tot:
            print(f"    {EL()} gptq({codebook}) {bits}b: {done}/{tot} layers", flush=True)

@torch.no_grad()
def perplexity(model, ids, ctx, dev):
    nll = ntok = 0
    for i in range(0, ids.size(1)-1, ctx):
        b = min(i+ctx, ids.size(1)-1)
        lo = model(ids[:, i:b].to(dev)).logits
        nll += torch.nn.functional.cross_entropy(
            lo.reshape(-1, lo.size(-1)), ids[:, i+1:b+1].reshape(-1).to(dev), reduction="sum").item()
        ntok += b - i
    return float(np.exp(nll / ntok))

### Run the full sweep over ALL models (checkpoints to CSV after every row)
One session does every model in `CONFIG["models"]` — each gets its own tokenizer, C4
calibration, and eval, and all rows land in the same CSV (the `model` column separates
them). A spot-instance kill only loses the in-flight row.

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoConfig, AutoModelForCausalLM

ALL_ROWS = []; DS_ROWS = []
def flush_csv():
    if not ALL_ROWS: return
    with open(CONFIG["out_csv"], "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(ALL_ROWS[0].keys())); w.writeheader(); w.writerows(ALL_ROWS)
def flush_ds():
    if not DS_ROWS: return
    cols = list({k for r in DS_ROWS for k in r})
    order = ["model","method","bits","group"] + [c for c in cols if c not in ("model","method","bits","group","avg")] + ["avg"]
    with open(CONFIG["ds_csv"], "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=order); w.writeheader(); w.writerows(DS_ROWS)

def downstream(m, tok, M, method, bits, group):
    """zero-shot accuracy via lm-eval, HEADLINE configs only (reference + ds_methods at ds_groups)."""
    if method != "reference" and (method not in CONFIG["ds_methods"] or group not in CONFIG["ds_groups"]): return
    from lm_eval.evaluator import simple_evaluate
    from lm_eval.models.huggingface import HFLM
    print(f"  {EL()} downstream {method} {bits}b g{group} ({','.join(CONFIG['ds_tasks'])})...", flush=True)
    lm = HFLM(pretrained=m, tokenizer=tok, batch_size="auto")
    res = simple_evaluate(model=lm, tasks=CONFIG["ds_tasks"], limit=(CONFIG["ds_limit"] or None),
                          bootstrap_iters=0, verbosity="ERROR")
    row = dict(model=M, method=method, bits=bits, group=group); accs=[]
    for t,d in res["results"].items():
        a = d.get("acc_norm,none", d.get("acc,none"))
        row[t] = round(a,4) if a is not None else ""
        if a is not None: accs.append(a)
    row["avg"] = round(sum(accs)/len(accs),4) if accs else ""
    DS_ROWS.append(row); flush_ds()
    print(f"  {EL()} downstream {method} {bits}b g{group}: avg={row['avg']}", flush=True)

def run_model(M):
    cfg = AutoConfig.from_pretrained(M)
    maxpos = getattr(cfg, "max_position_embeddings", None) or getattr(cfg, "n_positions", None)
    CTX = min(CONFIG["ctx"], maxpos) if maxpos else CONFIG["ctx"]
    tok = AutoTokenizer.from_pretrained(M, use_fast=True)
    print(f"\n===== {M}  (ctx {CTX}) =====", flush=True)

    def c4_tokens(ntok, split):
        ds = load_dataset("allenai/c4", "en", split=split, streaming=True)
        ch, have = [], 0
        for ex in ds:
            e = tok(ex["text"], return_tensors="pt").input_ids; ch.append(e); have += e.size(1)
            if have >= ntok + CTX: break
        ids = torch.cat(ch, 1); return ids[:, :((ntok // CTX) * CTX if ntok else ids.size(1))]

    wt = tok("\n\n".join(load_dataset("wikitext","wikitext-2-raw-v1",split="test")["text"]),
             return_tensors="pt").input_ids
    if CONFIG["eval_tokens"]: wt = wt[:, :CONFIG["eval_tokens"]]
    c4_eval = c4_tokens(CONFIG["eval_tokens"] or 262144, "validation") if CONFIG["eval_c4"] else None
    if CONFIG["calib_source"] == "c4":
        calib = c4_tokens(CONFIG["calib_tokens"], "train")
    else:
        calib = tok("\n\n".join(load_dataset("wikitext","wikitext-2-raw-v1",split="train")["text"][:8000]),
                    return_tensors="pt").input_ids[:, :CONFIG["calib_tokens"]]
    print(f"  wikitext2 eval tokens {wt.size(1)} | calib tokens {calib.size(1)}", flush=True)

    def load(): return AutoModelForCausalLM.from_pretrained(M, torch_dtype=DTYPE).eval().to(DEV)
    def rec(method, bits, group, pw, pc, base):
        d = 100*(pw-base)/base
        ALL_ROWS.append(dict(model=M, method=method, bits=bits, group=group,
                             wikitext2=round(pw,4), delta_pct=round(d,2), c4=round(pc,4) if pc else ""))
        print(f"  {EL()} {method:9s} {str(bits):>2}b g{group:<4} wikitext2={pw:8.3f} ({d:+.2f}%)"
              + (f"  c4={pc:.3f}" if pc else ""), flush=True); flush_csv()

    m = load(); base = perplexity(m, wt, CTX, DEV)
    bc = perplexity(m, c4_eval, CTX, DEV) if c4_eval is not None else None
    rec("reference","-",0,base,bc,base); downstream(m, tok, M, "reference", "-", 0); del m
    for bits in CONFIG["bits"]:
        for group in CONFIG["groups"]:
            for method in ("uniform","codebook","kmeans"):
                m = load(); quantize_rtn(m, method, bits, group, DEV)
                pw = perplexity(m, wt, CTX, DEV); pc = perplexity(m, c4_eval, CTX, DEV) if c4_eval is not None else None
                rec(method, bits, group, pw, pc, base); downstream(m, tok, M, method, bits, group); del m
            m = load(); print(f"  {EL()} [collecting Hessians {bits}b g{group}, {calib.size(1)//CTX} passes...]", flush=True)
            hess = collect_hessians(m, calib, CTX, DEV)   # CPU-resident dict
            del m
            if DEV == "cuda": torch.cuda.empty_cache()
            for cb, tag in (("nf","nf+gptq"), ("kmeans","km+gptq")):
                m = load()
                quantize_gptq(m, hess, bits, group, DEV, codebook=cb)   # streams per-layer from CPU
                pw = perplexity(m, wt, CTX, DEV); pc = perplexity(m, c4_eval, CTX, DEV) if c4_eval is not None else None
                rec(tag, bits, group, pw, pc, base); downstream(m, tok, M, tag, bits, group); del m
                if DEV == "cuda": torch.cuda.empty_cache()
            del hess

for M in CONFIG["models"]:
    run_model(M)
    if DEV == "cuda": torch.cuda.empty_cache()
print("\ndone ->", CONFIG["out_csv"])

### Read the results
```python
import pandas as pd; pd.read_csv(CONFIG["out_csv"])
```
**How to read it:** at each `(bits, group)`, compare `uniform` vs `codebook` vs `nf+gptq`
`delta_pct` (perplexity increase over fp16, lower is better). The paper's claim is that
non-uniform (`codebook`/`nf+gptq`) never loses to `uniform` at equal bitwidth — and the
margin should **widen at 3-bit and 2-bit**, where the AMX free-gather is most compelling.
Headline bit-width is decided from these measured margins.

In [ ]:
import pandas as pd; pd.read_csv(CONFIG["out_csv"])